# PySpark & la MLlib

## Récupération des données

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-titanic.git
!ls -lh dataset-titanic/

## Import des libraries nécessaires

In [ ]:
!pip install inflection pyspark

In [ ]:
from pathlib import Path

import inflection
import pyspark
import pyspark.ml.feature as MF
import pyspark.sql.functions as F

## Chargement des données

Le fichier csv est chargé dans un `Dataframe` spark.

In [ ]:
data_dir = Path("dataset-titanic")
csv_path = data_dir / "train.csv"

In [ ]:
spark = (pyspark.sql.SparkSession.builder.appName("Démonstration de PySpark")
                                         .getOrCreate())
df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

In [ ]:
passengers_count = df.count()

In [ ]:
print(passengers_count)

In [ ]:
df.show(5)

In [ ]:
df.describe().show()

In [ ]:
df.printSchema()

## Renommage des colonnes pour utiliser la convention Python du snake case

In [ ]:
df = df.toDF(*(inflection.underscore(c) for c in df.columns))
df.show(5)

## Exploration des données

In [ ]:
df.select("survived", "pclass", "embarked").show()

In [ ]:
df.groupBy("survived").count().show()

In [ ]:
df.groupBy("sex", "survived").count().show()

In [ ]:
df.groupBy("pclass", "survived").count().show()

## Préparation des données

### Valeurs nulles

In [ ]:
def null_value_count(df: pyspark.sql.DataFrame):
  null_columns_counts = []
  for k in df.columns:
    null_rows = df.where(F.col(k).isNull()).count()
    if null_rows:
      null_columns_counts.append((k, null_rows))
  return null_columns_counts

null_columns_count_list = null_value_count(df)
spark.createDataFrame(null_columns_count_list,
                      ['column_with_nulls', 'nulls_count']).show()

### Analyse de l'âge des passagers

In [ ]:
mean_age = df.select(F.mean('age')).collect()[0][0]
print(mean_age)

On pourrait utiliser cet âge moyen pour l'imputation des âges inconnus, mais nous allons voir comment aller un peu plus loin et calculer âge moyen par civilité.

In [ ]:
df = df.withColumn("title", F.regexp_extract(F.col("name"), r"([A-Za-z]+)\.", 1))
df.show()

In [ ]:
df.select("title").distinct().show()

In [ ]:
replacements = dict(Miss=["Mlle", "Ms"],
                    Mr=["Dr", "Major", "Capt", "Sir", "Don"],
                    Mrs=["Lady", "Countess", "Mme"],
                    Other=["Jonkheer", "Col", "Rev"])
df = df.replace({name: title
                 for title, names in replacements.items()
                 for name in names},
                subset=["title"])

In [ ]:
df.select("title").distinct().show()

In [ ]:
average_title_ages = df.groupby("title").agg(F.avg("age").alias("average_age"))
average_title_ages.show()

### Imputation de l'âge

On attribue l'âge moyen du groupe de passagers partageant le même titre.

In [ ]:
df = (df.join(average_title_ages, "title")
        .withColumn("age", F.when(F.col("age").isNull(),
                                  F.col("average_age"))
                            .otherwise(F.col("age")))
        .drop("average_age"))
df.show()

La caractéristique *embarked* (le port d'embarcation) a seulement deux valeurs manquantes. Nous allons imputer le mode pour ces deux valeurs.

In [ ]:
df.groupBy("embarked").count().show()

In [ ]:
df = df.na.fill(dict(embarked="S"))

La caractéristique *cabin* peut être supprimée : le numéro de cabine est quasi unique pour chaque passager qui est en cabine et va provoquer rapidement du sur-apprentissage.

In [ ]:
df = df.drop("cabin")

In [ ]:
df.printSchema()

### Ajout d'une nouvelle caractéristique.

La taille de la famille.

In [ ]:
df = df.withColumn("family_size", F.col("sib_sp") + F.col("parch") + 1)

In [ ]:
df.groupBy("family_size").count().show()

In [ ]:
df = df.withColumn("alone", (F.col("family_size") == 1).cast("long"))
df.show()

In [ ]:
df.columns

### Encodage de variables catégorielles

In [ ]:
indexers = [MF.StringIndexer(inputCol=column,
                             outputCol=f"{column}_index").fit(df)
            for column in ["sex", "embarked", "title"]]
pipeline = pyspark.ml.Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)
df.show()

In [ ]:
df.printSchema()

### Suppression des caractéristiques non numériques

In [ ]:
df = df.drop(*(name for name, dtype in df.dtypes if dtype == "string"))

In [ ]:
df.show()

### Création des vecteurs de caractéristiques

In [ ]:
feature = MF.VectorAssembler(
    inputCols=df.drop("survived", "passenger_id").columns, outputCol="features")
feature_vector = feature.transform(df)

In [ ]:
feature_vector.show()

### Séparation en ensembles d'entraînement et de test

In [ ]:
training_data, test_data = feature_vector.randomSplit([0.8, 0.2], seed=11)

## Apprentissage de modèles

### Régression logistique

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="survived", featuresCol="features")
lr_model = lr.fit(training_data)
lr_prediction = lr_model.transform(test_data)
lr_prediction.select("prediction", "survived", "features").show()
evaluator = pyspark.ml.evaluation.MulticlassClassificationEvaluator(
    labelCol="survived", predictionCol="prediction", metricName="accuracy")

# Évaluation
lr_accuracy = evaluator.evaluate(lr_prediction)
print(f"Exactitude d'une régression logistique : {lr_accuracy:.2f}")

### Forêt aléatoire

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rfc = RandomForestClassifier(labelCol="survived", featuresCol="features")
rfc_model = rfc.fit(training_data)
rfc_predictions = rfc_model.transform(test_data)
rfc_predictions.select("prediction", "survived", "features").show()

# Évaluation
rfc_accuracy = evaluator.evaluate(rfc_predictions)
print(f"Exactitude d'une forêt aléatoire : {rfc_accuracy:.2f}")

### Arbre boosté

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbtc = GBTClassifier(labelCol="survived", featuresCol="features", maxIter=10)
gbtc_model = gbtc.fit(training_data)
gbtc_predictions = gbtc_model.transform(test_data)
gbtc_predictions.select("prediction", "survived", "features").show()

# Évaluation
gbtc_accuracy = evaluator.evaluate(gbtc_predictions)
print(f"Exactitude d'un arbre boosté : {gbtc_accuracy:.2f}")

### Séparateur à vaste marge

In [ ]:
from pyspark.ml.classification import LinearSVC

svmc = LinearSVC(labelCol="survived", featuresCol="features")
svmc_model = svmc.fit(training_data)
svmc_predictions = svmc_model.transform(test_data)
svmc_predictions.select("prediction", "survived", "features").show()

# Évaluation
svmc_accuracy = evaluator.evaluate(svmc_predictions)
print(f"Exactitude d'un séparateur à vaste marge : {gbtc_accuracy:.2f}")

## Référence

https://spark.apache.org/docs/latest/ml-classification-regression.html